# Splitting and Imputing Data

Splits each user segment (personal / professional) into train, validation and test
at the user level, then imputes missing `vehicle_start_year` with a KNN imputer fit
on train only. The fitted imputer and encoders are reused on val and test so no
information leaks from the held-out sets into the fill values.


In [1]:
# Move the working directory up to the project root so the src package imports resolve
import os
import polars as pl
from pathlib import Path

os.chdir(Path(os.getcwd()).parent)
os.getcwd()

'c:\\Users\\Tomas\\Desktop\\Thesis Stuff\\Survival_Analysis_Thesis\\Coding'

In [2]:
from src.constants import paths_to_files_and_folders as const
from src.constants.columns import USER_ID_COL
from src.data_splitting import DataSplitter
from src.data_processing import DataProcessor

1309


c:\Users\Tomas\Desktop\Thesis Stuff\Survival_Analysis_Thesis\Coding\src\data_processing.py:163: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_with_intervals = df.join_asof(


1233


In [3]:
# Filtered interim files produced by the cleaning step, one per user segment
path_to_personal_filtered = const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv"
path_to_professional_filtered = const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv"

In [4]:
path_to_personal_filtered

WindowsPath('C:/Users/Tomas/Desktop/Thesis Stuff/Survival_Analysis_Thesis/Coding/Data/interim/personal_users_filtered.csv')

In [5]:
import src.constants.paths_to_files_and_folders as paths

# One DataSplitter per segment - each holds its own frame for splitting and imputation
data_personal = pl.read_csv(path_to_personal_filtered)
data_splitter_personal = DataSplitter(data_personal)

data_professional = pl.read_csv(path_to_professional_filtered)
data_splitter_professional = DataSplitter(data_professional)

In [6]:
# Personal segment - split, KNN-impute start year, and save the three CSVs
train_df, val_df, test_df = data_splitter_personal.prepare_dataset(data_personal,
                              train_size=0.8,
                              test_size=0.1,
                              val_size=0.1,
                              personal=True,
                              save_path=Path(paths.PATH_TO_INTERIM_DATA))

# Unique user counts per split - confirms the split is at the user level and reproducible
print(train_df[USER_ID_COL].n_unique(),
      val_df[USER_ID_COL].n_unique(),
      test_df[USER_ID_COL].n_unique())

107902 12226 12096
1861 233 233


In [7]:
# Professional segment - same pipeline on the professional splitter and frame
train_df_prof, val_df_prof, test_df_prof = data_splitter_professional.prepare_dataset(data_professional,
                              train_size=0.8,
                              test_size=0.1,
                              val_size=0.1,
                              personal=False,
                              save_path=Path(paths.PATH_TO_INTERIM_DATA))

print(train_df_prof[USER_ID_COL].n_unique(),
      val_df_prof[USER_ID_COL].n_unique(),
      test_df_prof[USER_ID_COL].n_unique())

27366 3766 5072
165 21 21


In [8]:
data_personal["churn_triggered"].sum()

1309